In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# === 샘플 문서 준비 ===
document = """Adapterz(어댑터즈)는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.
어댑터즈의 모든 교재는 5단 분석법이라는 독자적인 교수법으로 작성됩니다.
어댑터즈는 인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다."""

# === Chat Model 생성 ===
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

# === 프롬프트, 파서 ===
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 문서를 참고하여 질문에 한 문장으로 답하세요.\n\n[document]\n{document}"),
    ("human", "{question}")
])
parser = StrOutputParser()

In [ ]:
# === LCEL 체인 구성 ===
chain = prompt | model | parser

# === 단일 입력 실행 ===
result = chain.invoke({
    "document": document,
    "question": "어댑터즈는 어디에서 운영하는 서비스인가요?"
})
print(f"결과: {result}")

In [ ]:
# === 여러 질문을 한꺼번에 처리 ===
questions = [
    {"document": document, "question": "어댑터즈는 어디에서 운영하나요?"},
    {"document": document, "question": "어댑터즈의 교수법은 무엇인가요?"},
    {"document": document, "question": "어댑터즈는 어떤 분야의 교재를 제공하나요?"},
]

results = chain.batch(questions)
for i, r in enumerate(results, 1):
    print(f"Q{i}: {r}")

In [ ]:
# === 응답을 실시간으로 받기 ===
for chunk in chain.stream({
    "document": document,
    "question": "어댑터즈에 대해 자세히 설명해 주세요."
}):
    print(chunk, end="", flush=True)

In [ ]:
from langchain_core.runnables import RunnableLambda

# === 응답을 후처리하는 함수 ===
# 모델 응답 앞에 [어댑터즈 안내] 접두어를 붙입니다.
def add_prefix(text: str) -> str:
    return f"[어댑터즈 안내] {text}"

# === RunnableLambda로 변환 ===
prefix_runnable = RunnableLambda(add_prefix)

# === 체인 마지막에 연결 ===
extended_chain = prompt | model | parser | prefix_runnable

result = extended_chain.invoke({
    "document": document,
    "question": "어댑터즈는 어디에서 운영하나요?"
})
print(result)